In [ ]:
import cv2 as cv 

import tensorflow as tf 
from tensorflow.keras.layers import Conv2D, Input, Dense, MaxPool2D, BatchNormalization, GlobalAvgPool2D # type: ignore
from tensorflow.keras.optimizers import Adam # type: ignore
from tensorflow.keras.preprocessing.image  import ImageDataGenerator ,load_img # type: ignore
from tensorflow.keras.callbacks import    ModelCheckpoint ,EarlyStopping, History # type: ignore
from tensorflow.keras import Model # type: ignore
from tensorflow.keras import metrics # type: ignore
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import glob 
import os
import shutil
import PIL
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
batch_size = 32
img_hieght = 224
img_width  = 224

In [ ]:
source_dir = "/kaggle/input/breast-cancer-detection-challenge"

destination_dir = "/kaggle/working/breast-cancer-detection-challenge"

shutil.copytree(source_dir, destination_dir)

In [ ]:

import os

def remove_files(folder_path, index):
    files = os.listdir(folder_path)
    files.sort()  

    for i in range(index + 1, len(files)):
        file_path = os.path.join(folder_path, files[i])
        if os.path.isfile(file_path):
            os.remove(file_path)
            print(f"Removed: {file_path}")

folder_path = "/kaggle/working/breast-cancer-detection-challenge/data/train/0"
index = 900
remove_files(folder_path, index)

In [ ]:
data_for_training = pathlib.Path("/kaggle/working/breast-cancer-detection-challenge/data/train")
data_for_testing = pathlib.Path("/kaggle/working/breast-cancer-detection-challenge/data/test")

In [ ]:
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(data_for_training
                                                                , batch_size=32 
                                                                , image_size=(img_hieght,img_width)
                                                                ,label_mode='categorical'
                                                                ,seed=89,subset="training"
                                                                ,validation_split=0.2)

In [ ]:
val_dataset = tf.keras.preprocessing.image_dataset_from_directory(data_for_training
                                                                , batch_size=32 
                                                                , image_size=(img_hieght,img_width)
                                                                ,label_mode='categorical'
                                                                ,seed=89,
                                                                  subset="validation"
                                                                ,validation_split=0.2)

In [ ]:
input_shape=(img_hieght,img_width,3)

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, MaxPool2D, BatchNormalization, GlobalAvgPool2D, Dense
from tensorflow.keras.regularizers import l2

@tf.keras.utils.register_keras_serializable()
class soheib_Model_MKI(tf.keras.Model):
    def __init__(self):
        super().__init__()
        
        # Apply L2 regularization to Conv2D and Dense layers
        self.conv1 = Conv2D(32, (3,3), activation='relu', kernel_regularizer=l2(0.001))  # L2 regularization
        self.maxpool1 = MaxPool2D()
        self.batchnorm1 = BatchNormalization()

        self.conv2 = Conv2D(64, (3,3), activation='relu', kernel_regularizer=l2(0.001))  # L2 regularization
        self.maxpool2 = MaxPool2D()
        self.batchnorm2 = BatchNormalization()

        self.conv3 = Conv2D(128, (3,3), activation='relu', kernel_regularizer=l2(0.001))  # L2 regularization
        self.globalavgpool1 = GlobalAvgPool2D()

        self.dense1 = Dense(32, activation='relu', kernel_regularizer=l2(0.001))  # L2 regularization
        self.dense2 = Dense(2, activation='softmax')  # Output layer without regularization

    def call(self, my_input):
        x = self.conv1(my_input)
        x = self.maxpool1(x)
        x = self.batchnorm1(x)

        x = self.conv2(x)
        x = self.maxpool2(x)
        x = self.batchnorm2(x)

        x = self.conv3(x)
        x = self.globalavgpool1(x)

        x = self.dense1(x)
        x = self.dense2(x)

        return x

    def build(self, input_shape):
        super().build(input_shape)


In [ ]:
from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.image import ImageDataGenerator # type:ignore

from tensorflow.keras.layers import Conv2D, Input, Dense, MaxPool2D, BatchNormalization, GlobalAvgPool2D # type: ignore

from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping # type: ignore

from tensorflow.keras import Model 

In [ ]:
model = soheib_Model_MKI()

model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [ ]:
inputs = tf.keras.Input(shape=(224, 224, 3))
outputs = model(inputs)

model.summary()

In [ ]:
path_to_save_model = './Models/myModel.keras' 

ckpt_saver = ModelCheckpoint(
    path_to_save_model,
    monitor='val_accuracy', 
    mode = 'max', 
    save_best_only = True,
    save_freq='epoch',
    verbose=1
) 

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=10 
)

In [ ]:
epochs = 40
saved = model.fit(train_dataset,validation_data=val_dataset,epochs=epochs)

In [ ]:
acc = saved.history['accuracy']
val_acc = saved.history['val_accuracy']

loss = saved.history['loss']
val_loss = saved.history['val_loss']

epochs_range = range(epochs)

plt.figure(figsize=(8, 8))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')

plt.show()

In [ ]:
import pandas as pd
import numpy as np
from keras.preprocessing import image
from keras.models import load_model

df = pd.read_csv('/kaggle/working/breast-cancer-detection-challenge/sample_submissions.csv')



predictions = []

for img_path in df['file_name']:  
    img = image.load_img("/kaggle/working/breast-cancer-detection-challenge/data/test/" +img_path, target_size=(224, 224))  # Redimensionner l'image
    img_array = image.img_to_array(img) / 255.0 
    img_array = np.expand_dims(img_array, axis=0)  
    pred = model.predict(img_array)  
    predictions.append(1 if pred[0][0] > 0.69 else 0) 




df['label'] = predictions  


df.to_csv('/kaggle/working/breast-cancer-detection-challenge/sample_submissions.csv', index=False)